# Day 4 — Sweet Crumbs Home Bakery
Run in Google Colab. Add GROQ_API_KEY in Secrets and enable notebook access. Uses GPT-OSS 120B. All orders are workshop test data.

In [ ]:
%pip install -q langchain langchain-groq gspread

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=userdata.get("GROQ_API_KEY").strip(), temperature=0.2, max_retries=0)

## Task 1 — Your tools

In [ ]:
from langchain.tools import tool
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = dict.fromkeys(["monday", "tuesday", "wednesday", "thursday", "friday", "saturday"], "9 am to 8 pm")
HOURS["sunday"] = "10 am to 6 pm"

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Calculate total rupee price for a menu item, e.g. chocolate cake, qty 2."""
    item = item.lower().strip()
    if qty < 1:
        return "Quantity must be at least 1."
    if item not in MENU:
        return f"Not on the menu. Available items: {', '.join(MENU)}"
    return f"{qty} x {item} = Rs {MENU[item] * qty}"

@tool
def is_available(item: str) -> str:
    """Check whether an item, e.g. cupcake, is offered on the menu, not live stock."""
    return f"{item} is on the menu; stock is not verified." if item.lower().strip() in MENU else f"{item} is not on the menu."

@tool
def is_open(day: str) -> str:
    """Look up opening hours for a weekday, e.g. sunday."""
    return HOURS.get(day.lower().strip(), "Invalid weekday")

print(get_price.invoke({"item": "chocolate cake", "qty": 2}))
print(is_available.invoke({"item": "cupcake"}))
print(is_open.invoke({"day": "sunday"}))

## Task 2 — Your record: five messy messages

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
class Order(BaseModel):
    """A bakery pickup order."""
    name: str = Field(min_length=1)
    item: Literal["plum cake", "chocolate cake", "cupcake"]
    qty: int = Field(ge=1)
    date: str = Field(min_length=1, description="Pickup date as stated")
    eggless: bool

extractor = llm.with_structured_output(Order, method="function_calling")
messages = [
    "ravi here need 2 choc cakes eggless dec 24",
    "hey im priya 6 cupcakes with egg pickup dec 25",
    "arun 1 plum cake no egg dec 26 pls",
    "meena here book 3 chocolate cakes regular with egg dec 27",
    "sai wants 4 eggless cupcakes pickup dec 28"
]
for text in messages:
    try:
        record = extractor.invoke("Extract the order. Normalize menu names. " + text)
        print(text, "=>", record.model_dump_json())
    except Exception as e:
        print(type(e).__name__, str(e))
        if "429" in str(e):
            break

## Task 3 — Your sheet
Run authentication and approve access. Reuses a uniquely named workshop sheet; if you have multiple sheets with this name, use open_by_key instead.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
try:
    sh = gc.open("Sweet Crumbs Orders - Day 4 Test")
except gspread.SpreadsheetNotFound:
    sh = gc.create("Sweet Crumbs Orders - Day 4 Test")
ws = sh.sheet1
if not ws.row_values(1):
    ws.append_row(["name", "item", "qty", "date", "eggless", "total"])
print("Sheet ready:", sh.url)

In [ ]:
@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed pickup order with customer name, menu item, positive quantity, pickup date and eggless preference."""
    try:
        order = Order(name=name.strip(), item=item.lower().strip(), qty=qty, date=date.strip(), eggless=eggless)
        if order.date.lower() in {"unknown", "none", "not specified"}:
            return "Ask for pickup date."
    except ValueError as e:
        return f"Invalid order: {e}"
    total = MENU[order.item] * order.qty
    try:
        ws.append_row([order.name, order.item, order.qty, order.date, "yes" if order.eggless else "no", total], value_input_option="RAW")
    except Exception as e:
        return f"Save not confirmed: {type(e).__name__}. Check sheet before retrying."
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

from langchain.agents import create_agent
SYSTEM = """You are Sweet Crumbs Home Bakery's friendly assistant. Orders need 24 hours notice; all cakes have eggless versions. Use tools for prices, menu availability and hours. Never guess stock. Save only explicitly requested orders with name, item, positive quantity, pickup date and eggless preference. Ask if missing. Do not save invalid orders. Do not claim success unless the tool confirms it. Reply under 60 words in English or Telugu."""
agent = create_agent(model=llm, tools=[get_price, is_available, is_open, log_order], system_prompt=SYSTEM)
def ask(agent, text):
    return agent.invoke({"messages": [{"role": "user", "content": text}]}, config={"recursion_limit": 10})
# Run once to avoid duplicate test orders.
result = ask(agent, "I am Ravi. Please book 2 eggless chocolate cakes for 24 Dec.")
print(result["messages"][-1].content)
print(ws.get_all_values()[-1])

## Task 4 — Break it: compare the same question with a vague tool name

In [ ]:
@tool
def tool1(item: str, qty: int = 1) -> str:
    """does stuff"""
    return get_price.invoke({"item": item, "qty": qty})

# Neutral prompt avoids naming the correct tool in either condition.
comparison_prompt = "Answer bakery questions using the appropriate tool. Never guess prices."
good_agent = create_agent(model=llm, tools=[get_price, is_open, is_available], system_prompt=comparison_prompt)
broken_agent = create_agent(model=llm, tools=[tool1, is_open, is_available], system_prompt=comparison_prompt)
question = "How much are 2 chocolate cakes?"
for label, candidate in [("Clear", good_agent), ("Vague", broken_agent)]:
    run = ask(candidate, question)
    print(label, "calls:", [c["name"] for m in run["messages"] for c in getattr(m, "tool_calls", [])])
    print(run["messages"][-1].content)
# Original agent still uses get_price; no permanent rename.

## Task 5 — Read the trace

In [ ]:
from langchain_core.messages import AIMessage
trace = ask(agent, "Are you open on Sunday? How much are 2 chocolate cakes?")
for m in trace["messages"]:
    m.pretty_print()
print("Tool calls:", sum(len(getattr(m, "tool_calls", [])) for m in trace["messages"]))
print("Model response rounds:", sum(isinstance(m, AIMessage) for m in trace["messages"]))

## Additional negative tests — no invalid rows should be saved

In [ ]:
for question in ["Do you sell donuts?", "I am Ravi. Book 2 eggless cupcakes.", "I am Ravi. Book 0 eggless cupcakes for 24 Dec."]:
    before = len(ws.get_all_values())
    run = ask(agent, question)
    print(question, "=>", run["messages"][-1].content)
    print("No new row:", len(ws.get_all_values()) == before)

Append the three negative cases to your existing Day 3 tests.csv. Choose must_include values matching the actual expected wording; verify no row was written separately. Do not rerun all cells repeatedly because each agent run can make multiple API requests.